# Cost & usage dashboard

Read-only view of every metered LLM call in `runs/usage_log.jsonl` — where the
money went, by phase / run / source, and the token-amplification effect.

**Kernel:** needs `pandas` + `altair` (in `.venv`; `pip install -r requirements.txt`).
Charts use the brand palette via `scripts/viz_theme.py`. Read-only — it never
spends or mutates anything; `scripts/cost_report.py` is the authoritative cost tool.

In [ ]:
from pathlib import Path
import sys

def find_repo(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "runs" / "usage_log.jsonl").exists():
            return p
    raise RuntimeError("run this notebook from inside the repo")

REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts"))   # for viz_theme

import pandas as pd
import altair as alt
import viz_theme
viz_theme.register()                        # brand Altair theme (burnt-orange on cream)
P = viz_theme.PALETTE

df = pd.read_json(REPO / "runs" / "usage_log.jsonl", lines=True)
df["run"] = df.run_id.map(lambda r: f"{r[4:6]}-{r[6:8]} {r[9:11]}:{r[11:13]}")  # 07-15 17:47
print(f"{len(df):,} logged LLM calls across {df.run_id.nunique()} runs, "
      f"total ${df.cost_usd.sum():.2f}")

## A. Spend by phase

**Index _building_ is the real cost** ($13.67), not retrieval —
the LLM-inferred tree pass dwarfs `page_content` + `answer`. Deterministic index
builds (outline / Markdown headings) avoid this entirely.

In [ ]:
# spend is one series (magnitude) -> single burnt-orange hue, phase on the axis
a = df.groupby("phase", as_index=False).cost_usd.sum()
bars = alt.Chart(a).mark_bar().encode(
    x=alt.X("cost_usd:Q", title="spend (USD)"),
    y=alt.Y("phase:N", sort="-x", title=None),
    tooltip=["phase", alt.Tooltip("cost_usd:Q", format="$.2f")])
labels = bars.mark_text(align="left", dx=4, color=P["ink"]).encode(
    text=alt.Text("cost_usd:Q", format="$.2f"))
(bars + labels).properties(
    title="Where the money goes: index build dominates", width=460, height=180)

## B. Spend per run, by phase

Which runs cost what, split by phase. The tall
`index_build` cell is the one metered PDF build; retrieval runs are the cheaper rows.

In [ ]:
# 5 phases x many runs -> a sequential heatmap (one hue), never 5 categorical colors
b = df.groupby(["run", "phase"], as_index=False).cost_usd.sum()
b = b[b.cost_usd > 0]
alt.Chart(b).mark_rect(stroke=P["ground"], strokeWidth=2).encode(
    x=alt.X("phase:N", title=None),
    y=alt.Y("run:N", title=None,
            sort=alt.EncodingSortField("cost_usd", "sum", "descending")),
    color=alt.Color("cost_usd:Q", title="USD",
                    scale=alt.Scale(range=viz_theme.SEQUENTIAL)),
    tooltip=["run", "phase", alt.Tooltip("cost_usd:Q", format="$.3f")]
).properties(title="Spend per run, by phase", width=360)

## C. Calls by source

`litellm` = real paid calls, `sdk` = Agents-SDK retrieval,
`cache-replay` = served free from the resumable-build cache. Those replays are the
money you *didn't* re-spend when continuing an aborted build.

In [ ]:
# calls by source: single-hue magnitude bar (source on axis, count = length)
c = df.assign(source=df.source.fillna("none")).groupby("source", as_index=False).size()
bars = alt.Chart(c).mark_bar().encode(
    x=alt.X("size:Q", title="LLM calls"),
    y=alt.Y("source:N", sort="-x", title=None),
    tooltip=["source", "size"])
labels = bars.mark_text(align="left", dx=4, color=P["ink"]).encode(text="size:Q")
replays = int((df.source == "cache-replay").sum())
print(f"{replays} calls served from the response cache at $0 "
      f"(resumable-build replays — real spend avoided on re-runs).")
(bars + labels).properties(title="LLM calls by source", width=460, height=160)

## D. Token amplification

Inside one agentic retrieval loop, the prompt grows every
turn — the index tree (a tool result) stays in context and is re-billed as input each
call. This is why `content_tokens` alone undercounts the bill (see `usage_logging.py`).

In [ ]:
# token amplification within ONE agentic loop: prompt tokens climb every turn as
# the index tree is re-sent as input. Single series -> single hue, no legend.
ret_phases = ["structure", "page_content", "answer", "other"]
ret = df[df.phase.isin(ret_phases)].run_id.max()          # newest retrieval run
sub = df[(df.run_id == ret) & df.phase.isin(ret_phases)].copy()
idx, qid = sub.groupby(["index", "qid"]).size().idxmax()  # busiest loop
one = sub[(sub["index"] == idx) & (sub.qid == qid)].sort_values("call_idx")
one["prompt_sent"] = one.input_tokens + one.cache_read_tokens
alt.Chart(one).mark_line(point=True).encode(
    x=alt.X("call_idx:Q", title="agentic turn", axis=alt.Axis(tickMinStep=1)),
    y=alt.Y("prompt_sent:Q", title="prompt tokens sent"),
    tooltip=["call_idx", "phase", "prompt_sent"]
).properties(title=f"Token amplification — {qid} on {idx}", width=460, height=240)

---
_Reconcile against `.venv/bin/python scripts/cost_report.py` — same `cost_usd` sums._